# 02 — ECG (PTB-XL) Pipeline
Simplified merge of old `07, 08, 09, 10, 11` (+ `12` saliency summarized in 03).

**Sections:** 1. Exploration/labels/preprocessing summaries · 2. Splits · 3. CNN eval + live prediction

_No retraining — loads `data/processed`, `reports`, `models`, one sample record._

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt


## 1. Summaries (tolerate missing raw `scp_statements.csv`)

In [ ]:
for f in ['reports/ptbxl_exploration_summary.json',
          'reports/ptbxl_label_preparation_summary.json',
          'reports/ptbxl_ecg_preprocessing_summary.json']:
    try:
        d = json.load(open(f))
        print('---', f, '---')
        print(str(d)[:600])
    except FileNotFoundError:
        print('MISSING:', f)


## 2. Train/val/test splits

In [ ]:
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/processed/ptbxl_{split}.csv')
    print(split, df.shape, list(df.columns[:8]))


## 3. CNN evaluation + live prediction on sample record

In [ ]:
from src.models import load_ptbxl_model, load_ptbxl_thresholds
from src.prediction import load_ecg_record, predict_ptbxl
model = load_ptbxl_model()
thresholds = load_ptbxl_thresholds()
print('final metrics:', json.load(open('reports/ptbxl_cnn_final_metrics.json')))
display(pd.read_csv('reports/ptbxl_cnn_final_class_metrics.csv'))
signal = load_ecg_record('data/raw/ptb-xl/records100/00000/00001_lr')
print('signal:', signal.shape)
for cls, r in predict_ptbxl(model, signal, thresholds).items():
    print(f"{cls}: prob={r['probability']:.4f} positive={r['positive']}")


In [ ]:
cm = pd.read_csv('reports/ptbxl_cnn_final_class_metrics.csv')
plt.bar(cm.iloc[:, 0], cm['f1'] if 'f1' in cm.columns else cm.iloc[:, -1])
plt.title('PTB-XL per-class F1'); plt.xticks(rotation=15); plt.tight_layout(); plt.show()
